In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
PROJECT_ROOT = Path(
    r"C:\Users\DIPANSHU SHUKLA\OneDrive\Desktop\SIH_162_ML"
)

CLEANED_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed_data"
    / "firms_cleaned.csv"
)

EVENTS_DIR = PROJECT_ROOT / "data" / "events"
EVENTS_DIR.mkdir(parents=True, exist_ok=True)

print("Input:", CLEANED_FILE)
print("Events directory:", EVENTS_DIR)

Input: C:\Users\DIPANSHU SHUKLA\OneDrive\Desktop\SIH_162_ML\data\processed_data\firms_cleaned.csv
Events directory: C:\Users\DIPANSHU SHUKLA\OneDrive\Desktop\SIH_162_ML\data\events


In [3]:
df = pd.read_csv(CLEANED_FILE)

df["acq_date"] = pd.to_datetime(df["acq_date"])

print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()

Rows: 702968
Columns: 12


,latitude,longitude,acq_date,acq_time,satellite,brightness,confidence,frp,daynight,version,detections_same_location_date,source_file
0,8.98492,77.60124,2025-01-01,801,NOAA-20 VIIRS,341.70,n,8.35,D,2,1,DL_FIRE_J1V-C2_812115.zip
1,9.00014,77.61005,2025-01-01,801,NOAA-20 VIIRS,330.68,n,2.68,D,2,1,DL_FIRE_J1V-C2_812115.zip
2,9.27809,78.04165,2025-01-01,801,NOAA-20 VIIRS,330.24,n,1.58,D,2,1,DL_FIRE_J1V-C2_812115.zip
3,9.56098,77.63283,2025-01-01,1949,NOAA-21 VIIRS,303.48,n,0.98,N,2.0NRT,1,DL_FIRE_J2V-C2_812116.zip
4,9.56323,77.63605,2025-01-01,1949,NOAA-21 VIIRS,303.55,n,1.23,N,2.0NRT,1,DL_FIRE_J2V-C2_812116.zip


### Create observation timestamp

In [4]:
df["acq_time"] = (
    df["acq_time"]
    .astype(str)
    .str.zfill(4)
)

df["timestamp"] = pd.to_datetime(
    df["acq_date"].dt.strftime("%Y-%m-%d")
    + " "
    + df["acq_time"].str[:2]
    + ":"
    + df["acq_time"].str[2:4],
    errors="coerce"
)

print("Invalid timestamps:", df["timestamp"].isna().sum())

Invalid timestamps: 0


Sort observations

In [5]:
df = df.sort_values(
    ["latitude", "longitude", "timestamp"]
).reset_index(drop=True)

df.head()

,latitude,longitude,acq_date,acq_time,satellite,brightness,confidence,frp,daynight,version,detections_same_location_date,source_file,timestamp
0,8.07834,77.53704,2025-01-12,2030,NOAA-20 VIIRS,305.30,n,1.34,N,2,1,DL_FIRE_J1V-C2_812115.zip,2025-01-12 20:30:00
1,8.07930,77.53805,2025-01-11,2026,Suomi-NPP VIIRS,310.44,n,2.60,N,2,1,DL_FIRE_SV-C2_812117.zip,2025-01-11 20:26:00
2,8.07984,77.53673,2025-03-01,1942,NOAA-21 VIIRS,302.64,n,0.86,N,2.0NRT,1,DL_FIRE_J2V-C2_812116.zip,2025-03-01 19:42:00
3,8.07993,77.53718,2025-01-17,2037,NOAA-20 VIIRS,305.63,n,0.98,N,2,1,DL_FIRE_J1V-C2_812115.zip,2025-01-17 20:37:00
4,8.08028,77.53823,2025-03-01,2008,Suomi-NPP VIIRS,310.91,n,1.18,N,2,1,DL_FIRE_SV-C2_812117.zip,2025-03-01 20:08:00


Inspect temporal coverage

In [6]:
print("First observation:", df["timestamp"].min())
print("Last observation:", df["timestamp"].max())

First observation: 2025-01-01 03:50:00
Last observation: 2025-03-19 21:31:00


Inspect spatial coverage

In [7]:
print("Latitude:")
print("Min:", df["latitude"].min())
print("Max:", df["latitude"].max())

print("\nLongitude:")
print("Min:", df["longitude"].min())
print("Max:", df["longitude"].max())

Latitude:
Min: 8.07834
Max: 34.49413

Longitude:
Min: 68.5089
Max: 96.98654


Inspect observations per date

In [8]:
daily_counts = (
    df.groupby(df["timestamp"].dt.date)
      .size()
      .sort_index()
)

daily_counts.head(10)

timestamp
2025-01-01    3909
2025-01-02    4789
2025-01-03    5542
2025-01-04    4855
2025-01-05    3982
2025-01-06    3592
2025-01-07    3442
2025-01-08    4264
2025-01-09    4002
2025-01-10    3666
dtype: int64

In [9]:
daily_counts.describe()

count       78.000000
mean      9012.410256
std       7294.543898
min       1784.000000
25%       4210.000000
50%       6309.500000
75%       9049.750000
max      30928.000000
dtype: float64

Inspect satellite sources

In [11]:
df["satellite"].value_counts(dropna=False)

satellite
NOAA-20 VIIRS      228688
Suomi-NPP VIIRS    227069
NOAA-21 VIIRS      213487
Terra MODIS         33724
Name: count, dtype: int64

Inspect day/night observations

In [12]:
df["daynight"].value_counts(dropna=False)

daynight
D    493791
N    209177
Name: count, dtype: int64

Inspect FRP

In [13]:
df["frp"].describe()

count    702968.000000
mean          8.523616
std          24.600331
min           0.000000
25%           1.990000
50%           3.870000
75%           7.110000
max        2056.400000
Name: frp, dtype: float64

Check same-location/date information

In [15]:
df["detections_same_location_date"].describe()

count    702968.000000
mean          1.000009
std           0.002922
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max           2.000000
Name: detections_same_location_date, dtype: float64

Prepare observations for event construction

In [16]:
df["observation_id"] = np.arange(len(df))

print("Observation IDs created:", df["observation_id"].nunique())
print("Total observations:", len(df))

Observation IDs created: 702968
Total observations: 702968


Check timestamp ordering

In [17]:
is_sorted = df["timestamp"].is_monotonic_increasing

print("Globally timestamp sorted:", is_sorted)

Globally timestamp sorted: False


Create a timestamp-sorted view

In [18]:
df_time = df.sort_values(
    "timestamp"
).reset_index(drop=True)

print(df_time[[
    "observation_id",
    "latitude",
    "longitude",
    "timestamp",
    "frp",
    "satellite"
]].head(10))

   observation_id  latitude  longitude           timestamp   frp    satellite
0          669751   26.9361    94.0335 2025-01-01 03:50:00   6.5  Terra MODIS
1          666340   26.7587    92.7613 2025-01-01 03:51:00   5.3  Terra MODIS
2          620082   25.3231    91.6008 2025-01-01 03:51:00   8.0  Terra MODIS
3          620161   25.3249    91.5900 2025-01-01 03:51:00  11.3  Terra MODIS
4          656593   26.2760    92.8154 2025-01-01 03:51:00  11.6  Terra MODIS
5          477573   22.5703    85.8978 2025-01-01 03:52:00   3.9  Terra MODIS
6          562322   23.7843    85.5653 2025-01-01 03:52:00   5.6  Terra MODIS
7          559834   23.7789    86.2051 2025-01-01 03:52:00   7.1  Terra MODIS
8          557734   23.7744    86.2100 2025-01-01 03:52:00   5.9  Terra MODIS
9          528688   23.5216    87.8397 2025-01-01 03:52:00   3.5  Terra MODIS


In [19]:
print("df_time sorted:", df_time["timestamp"].is_monotonic_increasing)

df_time sorted: True


In [20]:
print("First timestamp:", df_time["timestamp"].min())
print("Last timestamp:", df_time["timestamp"].max())

print("\nObservations by day:")
print(
    df_time.groupby(df_time["timestamp"].dt.date)
    .size()
    .head(10)
)

First timestamp: 2025-01-01 03:50:00
Last timestamp: 2025-03-19 21:31:00

Observations by day:
timestamp
2025-01-01    3909
2025-01-02    4789
2025-01-03    5542
2025-01-04    4855
2025-01-05    3982
2025-01-06    3592
2025-01-07    3442
2025-01-08    4264
2025-01-09    4002
2025-01-10    3666
dtype: int64


temporal gaps

In [21]:
time_diff = df_time["timestamp"].diff()

print("Time gap statistics:")
print(time_diff.describe())

print("\nLargest gaps:")
print(
    time_diff.sort_values(ascending=False)
    .head(10)
)

Time gap statistics:
count                    702967
mean     0 days 00:00:09.554445
std      0 days 00:05:56.469168
min             0 days 00:00:00
25%             0 days 00:00:00
50%             0 days 00:00:00
75%             0 days 00:00:00
max             0 days 07:41:00
Name: timestamp, dtype: object

Largest gaps:
124459   0 days 07:41:00
36797    0 days 07:34:00
391965   0 days 07:33:00
125821   0 days 07:30:00
38377    0 days 07:28:00
324800   0 days 07:27:00
53490    0 days 07:26:00
153117   0 days 07:26:00
2356     0 days 07:24:00
188931   0 days 07:21:00
Name: timestamp, dtype: timedelta64[us]


zero-time gaps

In [22]:
zero_gap_count = (time_diff == pd.Timedelta(0)).sum()
positive_gap_count = (time_diff > pd.Timedelta(0)).sum()

print("Zero time gaps:", zero_gap_count)
print("Positive time gaps:", positive_gap_count)
print("Total gaps:", len(time_diff.dropna()))

Zero time gaps: 697066
Positive time gaps: 5901
Total gaps: 702967


In [23]:
positive_gaps = time_diff[time_diff > pd.Timedelta(0)]

print("Positive gap statistics:")
print(positive_gaps.describe())

Positive gap statistics:
count                      5901
mean     0 days 00:18:58.190137
std      0 days 01:02:02.256502
min             0 days 00:01:00
25%             0 days 00:01:00
50%             0 days 00:01:00
75%             0 days 00:03:00
max             0 days 07:41:00
Name: timestamp, dtype: object


In [24]:
zero_gap_count = (time_diff == pd.Timedelta(0)).sum()
positive_gap_count = (time_diff > pd.Timedelta(0)).sum()

print("Zero time gaps:", zero_gap_count)
print("Positive time gaps:", positive_gap_count)
print("Total gaps:", len(time_diff.dropna()))

Zero time gaps: 697066
Positive time gaps: 5901
Total gaps: 702967


geographic coverage


In [25]:
print("Latitude range:")
print(df_time["latitude"].min(), "to", df_time["latitude"].max())

print("\nLongitude range:")
print(df_time["longitude"].min(), "to", df_time["longitude"].max())

print("\nUnique coordinates:")
print(
    df_time[["latitude", "longitude"]]
    .drop_duplicates()
    .shape[0]
)

Latitude range:
8.07834 to 34.49413

Longitude range:
68.5089 to 96.98654

Unique coordinates:
702918


In [26]:
print("Spatial summary:")

print(
    df_time[["latitude", "longitude", "frp"]]
    .describe()
)

Spatial summary:
            latitude      longitude            frp
count  702968.000000  702968.000000  702968.000000
mean       20.838162      82.034182       8.523616
std         3.994711       5.942664      24.600331
min         8.078340      68.508900       0.000000
25%        18.474248      77.655307       1.990000
50%        21.214795      81.255650       3.870000
75%        23.495555      85.038702       7.110000
max        34.494130      96.986540    2056.400000


spatial duplicates

In [27]:
coordinate_counts = (
    df_time
    .groupby(["latitude", "longitude"])
    .size()
    .sort_values(ascending=False)
)

print("Top repeated coordinates:")
print(coordinate_counts.head(20))

Top repeated coordinates:
latitude  longitude
17.60906  83.20245     2
20.96270  85.17250     2
15.18193  76.67902     2
21.10460  72.64680     2
14.06651  77.85923     2
15.17398  76.65765     2
21.07951  85.03746     2
20.96368  85.17137     2
21.07688  85.03705     2
20.97951  85.13142     2
21.10179  72.63638     2
17.61007  83.20251     2
20.90899  71.45855     2
21.30940  81.59108     2
21.38848  81.64542     2
21.76354  84.01823     2
21.87483  83.49696     2
21.87925  83.55338     2
21.92498  83.34980     2
22.03930  83.35054     2
dtype: int64


In [28]:
print("Coordinates occurring more than once:")
print((coordinate_counts > 1).sum())

print("\nMaximum observations at one coordinate:")
print(coordinate_counts.max())

Coordinates occurring more than once:
50

Maximum observations at one coordinate:
2


### observations per day + satellite

In [31]:
# ============================================================
# CELL 24 — DAILY OBSERVATIONS BY SATELLITE
# ============================================================

daily_satellite = (
    df.groupby(["acq_date", "satellite"])
      .size()
      .reset_index(name="observation_count")
)

print("Daily satellite observation summary:")
display(daily_satellite.head(20))

print("\nNumber of satellite-date combinations:", len(daily_satellite))

Daily satellite observation summary:


,acq_date,satellite,observation_count
0,2025-01-01,NOAA-20 VIIRS,1205
1,2025-01-01,NOAA-21 VIIRS,1129
2,2025-01-01,Suomi-NPP VIIRS,1327
3,2025-01-01,Terra MODIS,248
4,2025-01-02,NOAA-20 VIIRS,1867
5,2025-01-02,NOAA-21 VIIRS,1192
6,2025-01-02,Suomi-NPP VIIRS,1603
7,2025-01-02,Terra MODIS,127
8,2025-01-03,NOAA-20 VIIRS,2062
9,2025-01-03,NOAA-21 VIIRS,1707



Number of satellite-date combinations: 312


### Satellite-wise summary

In [32]:
# ============================================================
# CELL 25 — SATELLITE SUMMARY
# ============================================================

satellite_summary = (
    df.groupby("satellite")
      .agg(
          observations=("observation_id", "count"),
          unique_locations=("observation_id", "nunique"),
          mean_frp=("frp", "mean"),
          median_frp=("frp", "median"),
          max_frp=("frp", "max"),
          mean_brightness=("brightness", "mean")
      )
      .reset_index()
)

display(satellite_summary)

,satellite,observations,unique_locations,mean_frp,median_frp,max_frp,mean_brightness
0,NOAA-20 VIIRS,228688,228688,7.195067,3.46,810.83,330.790355
1,NOAA-21 VIIRS,213487,213487,8.756581,4.21,1121.62,330.924377
2,Suomi-NPP VIIRS,227069,227069,7.256245,3.42,868.25,330.858176
3,Terra MODIS,33724,33724,24.591365,12.60,2056.40,321.138012


### Confidence distribution

In [33]:
# ============================================================
# CELL 26 — CONFIDENCE DISTRIBUTION
# ============================================================

print("Confidence values:")
print(df["confidence"].value_counts(dropna=False))

print("\nConfidence percentages:")
confidence_pct = (
    df["confidence"]
      .value_counts(normalize=True, dropna=False)
      .mul(100)
      .round(3)
)

display(confidence_pct)

Confidence values:
confidence
n     535352
l     112802
h      21090
69       945
64       936
       ...  
7          3
5          3
3          3
4          1
2          1
Name: count, Length: 103, dtype: int64

Confidence percentages:


confidence
n     76.156
l     16.047
h      3.000
69     0.134
64     0.133
       ...  
7      0.000
5      0.000
3      0.000
4      0.000
2      0.000
Name: proportion, Length: 103, dtype: float64

### FRP distribution

In [34]:
# ============================================================
# CELL 27 — FRP DISTRIBUTION
# ============================================================

frp_percentiles = df["frp"].quantile([
    0,
    0.25,
    0.50,
    0.75,
    0.90,
    0.95,
    0.99,
    0.999,
    1.0
])

display(frp_percentiles.to_frame("FRP"))

print("\nFRP > 50:", (df["frp"] > 50).sum())
print("FRP > 100:", (df["frp"] > 100).sum())
print("FRP > 500:", (df["frp"] > 500).sum())
print("FRP > 1000:", (df["frp"] > 1000).sum())

,FRP
0.000,0.0000
0.250,1.9900
0.500,3.8700
0.750,7.1100
0.900,13.8200
0.950,24.7300
0.990,103.4732
0.999,323.2400
1.000,2056.4000



FRP > 50: 16434
FRP > 100: 7374
FRP > 500: 178
FRP > 1000: 11


### Brightness distribution

In [35]:
# ============================================================
# CELL 28 — BRIGHTNESS DISTRIBUTION
# ============================================================

brightness_summary = df["brightness"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
)

display(brightness_summary.to_frame("brightness"))

,brightness
count,702968.000000
mean,330.389905
std,17.572496
min,207.380000
1%,297.130000
5%,301.290000
25%,313.660000
50%,334.860000
75%,341.950000
95%,354.170000


### Day/night × satellite

In [36]:
# ============================================================
# CELL 29 — DAY/NIGHT BY SATELLITE
# ============================================================

daynight_satellite = pd.crosstab(
    df["satellite"],
    df["daynight"],
    margins=True
)

display(daynight_satellite)

daynight,D,N,All
satellite,,,
NOAA-20 VIIRS,158888,69800,228688
NOAA-21 VIIRS,148277,65210,213487
Suomi-NPP VIIRS,157349,69720,227069
Terra MODIS,29277,4447,33724
All,493791,209177,702968


### Day/night FRP summary

In [37]:
# ============================================================
# CELL 30 — FRP BY DAY/NIGHT
# ============================================================

daynight_frp = (
    df.groupby("daynight")["frp"]
      .agg([
          "count",
          "mean",
          "median",
          "max"
      ])
      .reset_index()
)

display(daynight_frp)

,daynight,count,mean,median,max
0,D,493791,11.301802,5.30,2056.4
1,N,209177,1.965327,1.42,195.8


### Observations by hour

In [38]:
# ============================================================
# CELL 31 — OBSERVATIONS BY HOUR
# ============================================================

df["acq_hour"] = df["timestamp"].dt.hour

hourly_counts = (
    df.groupby("acq_hour")
      .size()
      .reset_index(name="observation_count")
)

display(hourly_counts)

,acq_hour,observation_count
0,2,16
1,3,796
2,4,4527
3,5,1781
4,6,54516
5,7,249972
6,8,169442
7,9,12724
8,10,17
9,14,245


### Acquisition time distribution

In [39]:
# ============================================================
# CELL 32 — ACQUISITION TIME SUMMARY
# ============================================================

df["acq_minute_of_day"] = (
    df["timestamp"].dt.hour * 60 +
    df["timestamp"].dt.minute
)

print("Earliest acquisition time:")
print(df["timestamp"].dt.time.min())

print("\nLatest acquisition time:")
print(df["timestamp"].dt.time.max())

print("\nMedian acquisition time:")
print(
    pd.to_timedelta(
        df["acq_minute_of_day"].median(),
        unit="m"
    )
)

Earliest acquisition time:
02:42:00

Latest acquisition time:
22:20:00

Median acquisition time:
0 days 08:09:00


### Same-location recurrence analysis

In [40]:
# ============================================================
# CELL 33 — LOCATION RECURRENCE
# ============================================================

location_counts = (
    df.groupby(["latitude", "longitude"])
      .size()
      .reset_index(name="observation_count")
)

print("Unique coordinates:", len(location_counts))

print("\nCoordinates with >= 2 observations:")
print((location_counts["observation_count"] >= 2).sum())

print("\nCoordinates with >= 3 observations:")
print((location_counts["observation_count"] >= 3).sum())

print("\nMaximum observations at one exact coordinate:")
print(location_counts["observation_count"].max())

Unique coordinates: 702918

Coordinates with >= 2 observations:
50

Coordinates with >= 3 observations:
0

Maximum observations at one exact coordinate:
2


### Multi-day recurrence

In [41]:
# ============================================================
# CELL 34 — MULTI-DAY LOCATION RECURRENCE
# ============================================================

location_days = (
    df.groupby(["latitude", "longitude"])["acq_date"]
      .nunique()
      .reset_index(name="unique_days")
)

print("Locations observed on >= 2 different days:",
      (location_days["unique_days"] >= 2).sum())

print("Locations observed on >= 3 different days:",
      (location_days["unique_days"] >= 3).sum())

print("Maximum number of different days at one exact coordinate:",
      location_days["unique_days"].max())

Locations observed on >= 2 different days: 47
Locations observed on >= 3 different days: 0
Maximum number of different days at one exact coordinate: 2


### Daily FRP statistics

In [42]:
# ============================================================
# CELL 35 — DAILY FRP STATISTICS
# ============================================================

daily_frp = (
    df.groupby("acq_date")["frp"]
      .agg(
          observation_count="count",
          total_frp="sum",
          mean_frp="mean",
          median_frp="median",
          max_frp="max"
      )
      .reset_index()
)

display(daily_frp.head(20))

,acq_date,observation_count,total_frp,mean_frp,median_frp,max_frp
0,2025-01-01,3909,13985.29,3.577716,2.660,59.60
1,2025-01-02,4789,20271.81,4.232994,3.110,115.71
2,2025-01-03,5542,26660.75,4.810673,3.275,155.60
3,2025-01-04,4855,28645.18,5.900140,3.470,389.24
4,2025-01-05,3982,16869.17,4.236356,2.990,183.48
5,2025-01-06,3592,14138.57,3.936128,2.875,46.40
6,2025-01-07,3442,14739.15,4.282147,2.980,146.00
7,2025-01-08,4264,18467.26,4.330971,2.900,153.00
8,2025-01-09,4002,16938.42,4.232489,2.870,93.89
9,2025-01-10,3666,15017.33,4.096380,2.940,59.52


### Daily high-FRP observations

In [43]:
# ============================================================
# CELL 36 — HIGH FRP BY DAY
# ============================================================

daily_high_frp = (
    df.assign(high_frp=df["frp"] >= 50)
      .groupby("acq_date")["high_frp"]
      .sum()
      .reset_index(name="high_frp_observations")
)

display(
    daily_high_frp.sort_values(
        "high_frp_observations",
        ascending=False
    ).head(20)
)

,acq_date,high_frp_observations
69,2025-03-11,2394
68,2025-03-10,1819
73,2025-03-15,1724
71,2025-03-13,1262
72,2025-03-14,1243
65,2025-03-07,1120
64,2025-03-06,811
70,2025-03-12,788
66,2025-03-08,652
62,2025-03-04,594


### emporal coverage summary

In [45]:
# ============================================================
# CELL 37 — TEMPORAL COVERAGE
# ============================================================

temporal_summary = {
    "first_timestamp": str(df["timestamp"].min()),
    "last_timestamp": str(df["timestamp"].max()),
    "number_of_days": int(df["acq_date"].nunique()),
    "number_of_observations": int(len(df)),
    "number_of_satellites": int(df["satellite"].nunique()),
}

temporal_summary

{'first_timestamp': '2025-01-01 03:50:00',
 'last_timestamp': '2025-03-19 21:31:00',
 'number_of_days': 78,
 'number_of_observations': 702968,
 'number_of_satellites': 4}

### Missing-value check

In [46]:
# ============================================================
# CELL 38 — REQUIRED FIELD COMPLETENESS
# ============================================================

required_columns = [
    "latitude",
    "longitude",
    "acq_date",
    "acq_time",
    "timestamp",
    "satellite",
    "brightness",
    "confidence",
    "frp",
    "daynight"
]

missing_summary = (
    df[required_columns]
    .isna()
    .sum()
    .to_frame("missing_count")
)

missing_summary["missing_percentage"] = (
    missing_summary["missing_count"] / len(df) * 100
).round(4)

display(missing_summary)

,missing_count,missing_percentage
latitude,0,0.0
longitude,0,0.0
acq_date,0,0.0
acq_time,0,0.0
timestamp,0,0.0
satellite,0,0.0
brightness,0,0.0
confidence,0,0.0
frp,0,0.0
daynight,0,0.0


### report

In [47]:
# ============================================================
# CELL 39 — FINAL NOTEBOOK 02 DIAGNOSTIC REPORT
# ============================================================

diagnostic_report = {
    "total_observations": int(len(df)),
    "unique_observations": int(df["observation_id"].nunique()),

    "first_timestamp": str(df["timestamp"].min()),
    "last_timestamp": str(df["timestamp"].max()),
    "unique_days": int(df["acq_date"].nunique()),

    "latitude_min": float(df["latitude"].min()),
    "latitude_max": float(df["latitude"].max()),
    "longitude_min": float(df["longitude"].min()),
    "longitude_max": float(df["longitude"].max()),

    "unique_coordinates": int(
        df[["latitude", "longitude"]].drop_duplicates().shape[0]
    ),

    "mean_frp": float(df["frp"].mean()),
    "median_frp": float(df["frp"].median()),
    "max_frp": float(df["frp"].max()),

    "zero_time_gaps": int(
        (df_time["timestamp"].diff() == pd.Timedelta(0)).sum()
    ),

    "positive_time_gaps": int(
        (df_time["timestamp"].diff() > pd.Timedelta(0)).sum()
    ),

    "repeated_exact_coordinates": int(
        (location_counts["observation_count"] >= 2).sum()
    ),

    "max_observations_exact_coordinate": int(
        location_counts["observation_count"].max()
    ),

    "satellites": df["satellite"].dropna().unique().tolist(),

    "daynight_values": df["daynight"].dropna().unique().tolist()
}

for key, value in diagnostic_report.items():
    print(f"{key}: {value}")

total_observations: 702968
unique_observations: 702968
first_timestamp: 2025-01-01 03:50:00
last_timestamp: 2025-03-19 21:31:00
unique_days: 78
latitude_min: 8.07834
latitude_max: 34.49413
longitude_min: 68.5089
longitude_max: 96.98654
unique_coordinates: 702918
mean_frp: 8.523616053646823
median_frp: 3.87
max_frp: 2056.4
zero_time_gaps: 697066
positive_time_gaps: 5901
repeated_exact_coordinates: 50
max_observations_exact_coordinate: 2
satellites: ['NOAA-20 VIIRS', 'Suomi-NPP VIIRS', 'NOAA-21 VIIRS', 'Terra MODIS']
daynight_values: ['N', 'D']


### Save diagnostic report

In [48]:
# ============================================================
# CELL 40 — SAVE DIAGNOSTIC REPORT
# ============================================================

import json
from pathlib import Path

PROJECT_ROOT = Path.cwd()
REPORT_DIR = PROJECT_ROOT / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

report_path = REPORT_DIR / "thermal_event_diagnostics.json"

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(
        diagnostic_report,
        f,
        indent=4,
        ensure_ascii=False
    )

print("Diagnostic report saved to:")
print(report_path)

Diagnostic report saved to:
c:\Users\DIPANSHU SHUKLA\OneDrive\Desktop\SIH_162_ML\notebooks\reports\thermal_event_diagnostics.json


### Save the prepared observation table

In [49]:
# ============================================================
# CELL 41 — SAVE PREPARED OBSERVATIONS
# ============================================================

OUTPUT_DIR = PROJECT_ROOT / "data" / "processed_data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

observation_output = OUTPUT_DIR / "firms_observations_prepared.csv"

df.to_csv(
    observation_output,
    index=False
)

print("Prepared observation data saved to:")
print(observation_output)

print("\nRows:", len(df))
print("Columns:", len(df.columns))

Prepared observation data saved to:
c:\Users\DIPANSHU SHUKLA\OneDrive\Desktop\SIH_162_ML\notebooks\data\processed_data\firms_observations_prepared.csv

Rows: 702968
Columns: 16
